Implementation of cuttle [cubic illuminance](https://www.researchgate.net/profile/Christopher-Cuttle/publication/274538293_Research_Note_A_practical_approach_to_cubic_illuminance_measurement/links/56a6d84b08ae997e22ba4fc6/Research-Note-A-practical-approach-to-cubic-illuminance-measurement.pdf)

In [31]:
#write the cubic points from the points.txt file --> done
import numpy as np
# make grid points for cubic illumination
class CubicfromPoints:
    "this"
    def __init__(self, input_file, space_angle=0, output_file=None):
        self.input_file = input_file
        self.space_angle = space_angle
        self.output_file = output_file or input_file.replace('points.txt', f'points_cubic_spaceAngle{space_angle}.txt')
        
        self.cubic_dtype = np.dtype([
            ('point', np.float64, (3,)),       # (x, y, z)
            ('x_plus', np.float64, (3,)),      # x+
            ('x_minus', np.float64, (3,)),     # x-
            ('y_plus', np.float64, (3,)),      # y+
            ('y_minus', np.float64, (3,)),     # y-
            ('z_plus', np.float64, (3,)),      # z+
            ('z_minus', np.float64, (3,)),     # z-
            ('u_plus', np.float64, (3,)),      # u+
            ('u_minus', np.float64, (3,)),     # u-
            ('v_plus', np.float64, (3,)),      # v+
            ('v_minus', np.float64, (3,)),     # v-
            ('w_plus', np.float64, (3,)),      # w+
            ('w_minus', np.float64, (3,))      # w-
        ])
        
        self.rotation_matrix = self._compute_rotation_matrix()
        self.data_array = None
    def _compute_rotation_matrix(self):
        theta = -np.pi / 4
        phi = np.arctan(np.sqrt(2))

        R_y = np.array([
            [np.cos(theta), 0, np.sin(theta)],
            [0, 1, 0],
            [-np.sin(theta), 0, np.cos(theta)]
        ])
        
        R_x = np.array([
            [1, 0, 0],
            [0, np.cos(phi), -np.sin(phi)],
            [0, np.sin(phi), np.cos(phi)]
        ])
        
        return R_x @ R_y
    def _read_input(self):
        with open(self.input_file, 'r') as f:
            lines = f.readlines()
            data = []
            for line in lines:
                values = line.strip().split()
                x, y, z, *_ = map(float, values)
                
                x_plus = np.array([np.cos(self.space_angle), np.sin(self.space_angle), 0])
                x_minus = np.array([-np.cos(self.space_angle), -np.sin(self.space_angle), 0])
                y_plus = np.array([-np.sin(self.space_angle), np.cos(self.space_angle), 0])
                y_minus = np.array([np.sin(self.space_angle), -np.cos(self.space_angle), 0])
                z_plus = np.array([0, 0, 1])
                z_minus = np.array([0, 0, -1])
                
                u_plus = self.rotation_matrix @ x_plus
                u_minus = self.rotation_matrix @ x_minus
                v_plus = self.rotation_matrix @ y_plus
                v_minus = self.rotation_matrix @ y_minus
                w_plus = self.rotation_matrix @ z_plus
                w_minus = self.rotation_matrix @ z_minus
                
                data.append((np.array([x, y, z]), x_plus, x_minus, y_plus, y_minus, z_plus, z_minus,
                             u_plus, u_minus, v_plus, v_minus, w_plus, w_minus))
        
        self.data_array = np.array(data, dtype=self.cubic_dtype)
    def write_output(self):
        if self.data_array is None:
            self._read_input()
        
        with open(self.output_file, 'w') as f:
            for i in self.data_array:
                x = i['point'][0]
                y = i['point'][1]
                z = i['point'][2]
                f.write(f"{x} {y} {z} {i['u_plus'][0]} {i['u_plus'][1]} {i['u_plus'][2]}\n")
                f.write(f"{x} {y} {z} {i['u_minus'][0]} {i['u_minus'][1]} {i['u_minus'][2]}\n")
                f.write(f"{x} {y} {z} {i['v_plus'][0]} {i['v_plus'][1]} {i['v_plus'][2]}\n")
                f.write(f"{x} {y} {z} {i['v_minus'][0]} {i['v_minus'][1]} {i['v_minus'][2]}\n")
                f.write(f"{x} {y} {z} {i['w_plus'][0]} {i['w_plus'][1]} {i['w_plus'][2]}\n")
                f.write(f"{x} {y} {z} {i['w_minus'][0]} {i['w_minus'][1]} {i['w_minus'][2]}\n")

class cubic_illumination:
    def __init__(self, E_u_plus, E_u_minus, E_v_plus, E_v_minus, E_w_plus, E_w_minus):
        self.E_u_plus = E_u_plus
        self.E_u_minus = E_u_minus
        self.E_v_plus = E_v_plus
        self.E_v_minus = E_v_minus
        self.E_w_plus = E_w_plus
        self.E_w_minus = E_w_minus
        # outputs
        # illumination vector components
        E_u = self.E_u_plus - self.E_u_minus
        E_v = self.E_v_plus - self.E_v_minus
        E_w = self.E_w_plus - self.E_w_minus
        # illumination vector magnitudes
        abs_E = np.sqrt(E_u**2 + E_v**2 + E_w**2) # |E|
        # symmetric components
        m_E_u = (self.E_u_plus + self.E_u_minus - E_u)/2 # ~E_u
        m_E_v = (self.E_v_plus + self.E_v_minus - E_v)/2 # ~E_v
        m_E_w = (self.E_w_plus + self.E_w_minus - E_w)/2 # ~E_w
        # symmetric illuminance
        m_E = (m_E_u + m_E_v + m_E_w)/3 # ~E
        # scalar illuminance 
        Esr = m_E + abs_E/4 # Esr
        # unit vectors
        e_u = E_u/abs_E
        e_v = E_v/abs_E
        e_w = E_w/abs_E
        e_x = 0.707*(e_v - e_w)
        e_y = 0.816*e_u - 0.408*(e_v + e_w)
        e_z = 0.577*(e_u + e_v + e_w)
        # Horizontal working plane illuminance
        if e_z < 0:
            Ewp = m_E
        else:
            Ewp = abs_E*e_z + m_E
        # mean cylindrical illuminance
        Ecl = (abs_E*(np.sqrt(e_x**2 + e_y**2))/np.pi) + m_E
        
        self.E_u = E_u
        self.E_v = E_v
        self.E_w = E_w
        self.abs_E = abs_E
        self.m_E_u = m_E_u
        self.m_E_v = m_E_v
        self.m_E_w = m_E_w
        self.m_E = m_E
        self.Esr = Esr
        self.m_E = m_E
        self.Ewp = Ewp
        self.Ecl = Ecl

In [ ]:
# make the cubic illumination points.txt
import os
spaces = {"S01", "S02", "S04", "S09", "S07", "S08", "S09", "S10", "S11"}
# change current working directory to /scratch/DelftBlueFiles/Inputs
for space in spaces:
    input_file = rf"/scratch/nforouzandehsh/DelftBlueFiles/Inputs/{space}/points.txt"
    # check if the file exists
    print(input_file)
    print(os.path.exists(input_file))
    CubicfromPoints(input_file, space_angle=0).write_output()

In [23]:
# example
E_u_plus = 10
E_u_minus = 230
E_v_plus = 320
E_v_minus = 50
E_w_plus = 40
E_w_minus = 530
CI = cubic_illumination(E_u_plus, E_u_minus, E_v_plus, E_v_minus, E_w_plus, E_w_minus)
print(CI.Ecl)

443.4042998137892


/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S01/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S09/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S08/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S11/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S10/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S07/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S02/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S04/points.txt
True
